# SysID Workflow

This notebook is the system-identification entry point for CloudPendulum. It assumes the folder starts with only `sysid_config.json` plus the Python scripts. Running the full workflow collects fresh npz data on the configured hardware cell, fits the model, and writes a new `identified_params_{cell}_fit_{timestamp}.json` file.

## Operating Rule

- New hardware cell: collect new npz data and fit from that data.
- Do not reuse npz or json files from another cell.
- Keep `sysid_config.json`; it is the only required initial configuration file.
- The default path below runs the complete hardware SysID pipeline.

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys
import time

ROOT = Path.cwd()
if (ROOT / "run_all.py").exists():
    SYSID_DIR = ROOT
elif (ROOT / "sysid_pipeline" / "run_all.py").exists():
    SYSID_DIR = ROOT / "sysid_pipeline"
else:
    SYSID_DIR = ROOT

if not (SYSID_DIR / "run_all.py").exists():
    raise FileNotFoundError("Run this notebook from the sysid_pipeline folder.")

print(f"SysID root: {SYSID_DIR}")

SysID root: /home/jupyter-venky/underactuated_double_pendulum_package


## Settings

`RUN_FULL_SYSID=True` starts hardware collection. This is intentional: the fresh-cell workflow must create new npz data before fitting.

In [2]:
# Full new-cell workflow: RUN0 -> RUN1 -> RUN2 collect fresh npz -> RUN3 fit fresh json.
RUN_FULL_SYSID = True

# For a complete new-cell identification, keep this as "all".
COLLECT_STAGE = "all"

# Normally keep this False. Set True only when the environment has already passed self-check.
SKIP_SELFCHECK = False

# Optional parameter subset override. Empty means sysid_common.FIT_NAMES.
# Example: FIT_NAMES_OVERRIDE = ["P1", "P2", "P3", "P6", "P4", "P5", "b1", "b2", "cf1", "cf2", "d1", "d2"]
FIT_NAMES_OVERRIDE = []

## Helpers

In [3]:
def load_config():
    cfg_path = SYSID_DIR / "sysid_config.json"
    if not cfg_path.exists():
        raise FileNotFoundError("Missing sysid_config.json. Create it or run sysid_make_config.py first.")
    with cfg_path.open("r", encoding="utf-8") as f:
        return json.load(f)


def data_root(cfg):
    return (SYSID_DIR / cfg.get("data_dir", ".")).resolve()


def run_step(tag, args, check=True):
    cmd = [sys.executable, *map(str, args)]
    shown = " ".join([Path(sys.executable).name, *map(str, args)])
    print("\n" + "=" * 72)
    print(f"{tag}: {shown}")
    print("=" * 72, flush=True)
    env = os.environ.copy()
    env.setdefault("PYTHONIOENCODING", "utf-8")
    t0 = time.perf_counter()
    proc = subprocess.Popen(
        cmd,
        cwd=str(SYSID_DIR),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        env=env,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()
    print(f"\nexit={rc} [{time.perf_counter() - t0:.1f}s]")
    if check and rc != 0:
        raise RuntimeError(f"{tag} failed with exit code {rc}")
    return rc


cfg = load_config()
print(json.dumps({k: ('<token>' if k == 'user_token' and v else v) for k, v in cfg.items()}, indent=2))

{
  "user_token": "<token>",
  "data_dir": ".",
  "experiment_type": "DoublePendulum",
  "cell_id": 203,
  "tau_max_id": 0.06,
  "dq_abort": 40.0,
  "ctrl_hz": 200.0,
  "inter_run_cooldown": 35.0,
  "start_retry_max": 4,
  "fit_data": [],
  "design_params": ""
}


## Preflight Asset Check

This check is informational. Existing npz/json files are not required. The full workflow below will create fresh files.

In [4]:
cfg = load_config()
root = data_root(cfg)
cell = cfg.get("cell_id", "*")
npz_before = sorted(root.glob(f"system_id_data_{cell}_*.npz"), key=lambda p: p.stat().st_mtime)
param_before = sorted(root.glob(f"identified_params_{cell}_*.json"), key=lambda p: p.stat().st_mtime)

print(f"cell_id: {cell}")
print(f"data root: {root}")
print(f"existing npz for this cell: {len(npz_before)}")
print(f"existing parameter json for this cell: {len(param_before)}")
print("The next cell will collect fresh npz when RUN_FULL_SYSID=True.")

cell_id: 203
data root: /home/jupyter-venky/underactuated_double_pendulum_package
existing npz for this cell: 0
existing parameter json for this cell: 0
The next cell will collect fresh npz when RUN_FULL_SYSID=True.


## Run Full System Identification

This is the main workflow cell. It calls `run_all.py --collect --stage all`, which collects new npz data and passes only the newly created npz files into RUN3.

In [5]:
if not RUN_FULL_SYSID:
    raise RuntimeError("RUN_FULL_SYSID=False. For a fresh-cell workflow, set it to True.")

cmd = ["run_all.py", "--collect", "--stage", COLLECT_STAGE]
if SKIP_SELFCHECK:
    cmd.append("--skip-selfcheck")
if FIT_NAMES_OVERRIDE:
    cmd.extend(["--fit-names", *FIT_NAMES_OVERRIDE])

run_step("Full SysID", cmd)


Full SysID: python run_all.py --collect --stage all

════════════════════════════════════════════════════════════════
  RUN 0  config  ->  python sysid_make_config.py 
════════════════════════════════════════════════════════════════
sysid_config.json already exists(token=185...248, len=19); not overwriting. To rebuild add --force.
  ✓ done  [0.5s]

════════════════════════════════════════════════════════════════
  RUN 1  offline self-check  ->  python sysid_run1_selfcheck.py 
════════════════════════════════════════════════════════════════

******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt
******************************************************************************

identification complete: 799  steps/10 windows, exact Hessian, 10

RuntimeError: Full SysID failed with exit code 1

## Outputs

The newest npz and parameter json files for the configured cell are shown below.

In [ ]:
cfg = load_config()
root = data_root(cfg)
cell = cfg.get("cell_id", "*")

npz_files = sorted(root.glob(f"system_id_data_{cell}_*.npz"), key=lambda p: p.stat().st_mtime)
param_files = sorted(root.glob(f"identified_params_{cell}_fit_*.json"), key=lambda p: p.stat().st_mtime)

print("Latest npz files:")
for p in npz_files[-5:]:
    print(f"  {p.name}")

if not param_files:
    raise RuntimeError("No identified_params_{cell}_fit_*.json was generated.")

latest = param_files[-1]
params = json.loads(latest.read_text(encoding="utf-8"))
print(f"\nLatest parameter json: {latest.name}")
meta = params.get("_meta", {})
if meta:
    print("meta:")
    print(json.dumps(meta, indent=2, ensure_ascii=False))

print("\nparameters:")
for k, v in params.items():
    if k != "_meta":
        print(f"  {k:>5} = {float(v):.8g}")